# Hidden-states dataset generation

Sample problems from the **MATH** benchmark, run them through a generative HF model
(Qwen or similar), and extract features from configurable layers:

- **pooled hidden states** — one vector per `(layer, pooling)` entry
- **geometric features** — scalars describing the shape/spread of the per-token hidden states at a layer

Each output row = one problem. Runs on a **Colab GPU** runtime (`Runtime -> Change runtime type -> GPU`).

## 1. Install dependencies

In [1]:
!pip -q install "transformers>=4.44" "datasets>=2.20" accelerate pyyaml pyarrow

<cell_type>markdown</cell_type>## 2. Config

Either use the hardcoded `CONFIG` dict below, **or** load a YAML file (set `USE_YAML = True`).

`layer` indexes `output_hidden_states`: `0` = embeddings, `1..N` = block outputs, negatives allowed (`-1` = last).

- **`features`** — pooled vectors. `pooling` is one of `mean | std | max | last | sum`.
- **`geometry`** — scalar shape statistics of the per-token hidden states (the "token cloud") at a layer.
  `metric` is one of `l2_norm | mean_token_norm | token_norm_std | anisotropy | effective_rank`.

`max_input_tokens` caps how many tokens of each problem are encoded (input truncation only — nothing is generated).

<details><summary>Example <code>config.yaml</code></summary>

```yaml
model_name: "Qwen/Qwen2.5-0.5B"
dataset_name: "EleutherAI/hendrycks_math"
dataset_config: "algebra"
dataset_split: "test"
num_problems: 200
shuffle_seed: 42
prompt_template: "{problem}"
features:
  - {layer: 3,  pooling: max}
  - {layer: 5,  pooling: last}
  - {layer: -1, pooling: mean}
geometry:
  - {layer: 3,  metric: l2_norm}
  - {layer: 5,  metric: mean_token_norm}
  - {layer: 5,  metric: anisotropy}
  - {layer: -1, metric: effective_rank}
batch_size: 8
max_input_tokens: 512
dtype: "float16"
device: "cuda"
output_path: "hidden_states_dataset.parquet"
```
</details>

In [2]:
import yaml

USE_YAML = True          # set True to load from a file instead of the dict below
YAML_PATH = "/content/hidden_states_config.yaml"

CONFIG = {
    # HuggingFace model id (loaded with AutoModel / AutoTokenizer)
    "model_name": "Qwen/Qwen2.5-0.5B",

    # MATH benchmark source + sampling
    "dataset_name": "EleutherAI/hendrycks_math",  # alt: 'lighteval/MATH', 'hendrycks/competition_math'
    "dataset_config": "algebra",                   # subject; None if the dataset has no config
    "dataset_split": "test",
    "num_problems": 200,
    "shuffle_seed": 42,
    "prompt_template": "{problem}",

    # pooled hidden-state vectors: (layer, pooling)
    "features": [
        {"layer": 3,  "pooling": "max"},
        {"layer": 5,  "pooling": "last"},
        {"layer": -1, "pooling": "mean"},
    ],

    # geometric scalars of the per-token hidden states: (layer, metric)
    "geometry": [
        {"layer": 3,  "metric": "l2_norm"},
        {"layer": 5,  "metric": "mean_token_norm"},
        {"layer": 5,  "metric": "anisotropy"},
        {"layer": -1, "metric": "effective_rank"},
    ],

    # runtime
    "batch_size": 8,
    "max_input_tokens": 512,  # input truncation length (NOT generation)
    "dtype": "float16",        # float16 | bfloat16 | float32
    "device": "cuda",          # cuda | cpu
    "output_path": "hidden_states_dataset.parquet",
}

if USE_YAML:
    with open(YAML_PATH) as f:
        CONFIG = yaml.safe_load(f)

CONFIG

{'model_name': 'Qwen/Qwen2.5-Math-1.5B',
 'dataset_name': 'EleutherAI/hendrycks_math',
 'dataset_config': 'algebra',
 'dataset_split': 'test',
 'num_problems': 200,
 'shuffle_seed': 42,
 'prompt_template': '{problem}',
 'layer_fractions': {'mid': 0.5, 'late': 0.75, 'final': 1.0},
 'features': [{'layer': 14, 'pooling': 'mean'},
  {'layer': 21, 'pooling': 'mean'},
  {'layer': -1, 'pooling': 'last'}],
 'geometry': [{'layer': 14, 'metric': 'l2_norm'},
  {'layer': 14, 'metric': 'token_norm_std'},
  {'layer': 21, 'metric': 'mean_token_norm'},
  {'layer': 21, 'metric': 'effective_rank'},
  {'layer': -1, 'metric': 'l2_norm'},
  {'layer': -1, 'metric': 'anisotropy'}],
 'batch_size': 8,
 'max_input_tokens': 512,
 'dtype': 'bfloat16',
 'device': 'cuda',
 'output_path': 'hidden_states_dataset.parquet'}

## 3. Load model & tokenizer

In [3]:
import torch
from transformers import AutoModel, AutoTokenizer

DTYPES = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}
device = CONFIG["device"] if torch.cuda.is_available() else "cpu"
dtype = DTYPES[CONFIG["dtype"]] if device == "cuda" else torch.float32
print(f"Device: {device} | dtype: {dtype}")

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModel.from_pretrained(
    CONFIG["model_name"],
    torch_dtype=dtype,
    output_hidden_states=True,
).to(device).eval()

num_layers = model.config.num_hidden_layers
print(f"Loaded {CONFIG['model_name']} | hidden_size={model.config.hidden_size} | layers={num_layers}")
print(f"hidden_states tuple length = {num_layers + 1} (index 0 = embeddings)")

Device: cuda | dtype: torch.bfloat16


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/676 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.32k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded Qwen/Qwen2.5-Math-1.5B | hidden_size=1536 | layers=28
hidden_states tuple length = 29 (index 0 = embeddings)


## 4. Sample problems from MATH

In [4]:
from datasets import load_dataset

ds = load_dataset(
    CONFIG["dataset_name"],
    CONFIG["dataset_config"],
    split=CONFIG["dataset_split"],
)

n = min(CONFIG["num_problems"], len(ds))
ds = ds.shuffle(seed=CONFIG["shuffle_seed"]).select(range(n))

# MATH rows expose a 'problem' field (and usually 'solution'/'level'/'type').
problems = [CONFIG["prompt_template"].format(problem=row["problem"]) for row in ds]
print(f"Sampled {len(problems)} problems.")
print("---\nExample:\n", problems[0][:400])

README.md:   0%|          | 0.00/3.93k [00:00<?, ?B/s]

algebra/train-00000-of-00001.parquet:   0%|          | 0.00/505k [00:00<?, ?B/s]

algebra/test-00000-of-00001.parquet:   0%|          | 0.00/353k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1744 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1187 [00:00<?, ? examples/s]

Sampled 200 problems.
---
Example:
 Square A and Square B are both $2009$ by $2009$ squares.  Square A has both its length and width increased by an amount $x$, while Square B has its length and width decreased by the same amount $x$.  What is the minimum value of $x$ such that the difference in area between the two new squares is at least as great as the area of a $2009$ by $2009$ square?


## 5. Feature functions

**Pooling** collapses the token cloud `(T, H)` into one vector `(H,)`.
**Geometry** describes the cloud's shape and returns a scalar. Both respect the attention mask
so padding tokens never contribute.

In [ ]:
def pool(hidden, mask, kind):
    """hidden: (B, T, H) float, mask: (B, T) {0,1}. Returns (B, H)."""
    m = mask.unsqueeze(-1).to(hidden.dtype)          # (B, T, 1)
    if kind == "mean":
        return (hidden * m).sum(1) / m.sum(1).clamp(min=1)
    if kind == "sum":
        return (hidden * m).sum(1)
    if kind == "std":
        # per-dimension standard deviation across valid tokens
        counts = m.sum(1).clamp(min=1)               # (B, 1)
        mean = (hidden * m).sum(1) / counts          # (B, H)
        var = ((hidden - mean.unsqueeze(1)) ** 2 * m).sum(1) / counts
        return var.clamp(min=0).sqrt()
    if kind == "max":
        neg = torch.finfo(hidden.dtype).min
        return hidden.masked_fill(m == 0, neg).max(1).values
    if kind == "last":
        idx = mask.sum(1).long() - 1                  # last non-pad token per row
        return hidden[torch.arange(hidden.size(0)), idx]
    raise ValueError(f"unknown pooling: {kind}")


def geometry(hidden, mask, metric):
    """Scalar shape statistic of the token cloud. hidden: (B, T, H) -> (B,)."""
    counts = mask.sum(1).clamp(min=1)                 # (B,) valid tokens per row
    h = hidden.float()
    fmask = mask.float()

    if metric == "l2_norm":
        # norm of the mean-pooled vector (how far the cloud's center sits from origin)
        mean_vec = (h * fmask.unsqueeze(-1)).sum(1) / counts.unsqueeze(-1)
        return mean_vec.norm(dim=-1)

    token_norms = h.norm(dim=-1)                      # (B, T) length of each token vector
    if metric == "mean_token_norm":
        return (token_norms * fmask).sum(1) / counts
    if metric == "token_norm_std":
        mean = ((token_norms * fmask).sum(1) / counts).unsqueeze(1)
        var = (((token_norms - mean) ** 2) * fmask).sum(1) / counts
        return var.clamp(min=0).sqrt()
    if metric == "anisotropy":
        # mean pairwise cosine similarity between distinct token vectors
        # = (||sum u_i||^2 - T) / (T*(T-1)),  u_i = unit token vectors
        u = torch.nn.functional.normalize(h, dim=-1) * fmask.unsqueeze(-1)
        sq = (u.sum(1) ** 2).sum(-1)
        denom = (counts * (counts - 1)).clamp(min=1)
        return (sq - counts) / denom
    if metric == "effective_rank":
        # participation ratio of the centered token cloud's singular spectrum:
        # (sum lambda)^2 / sum lambda^2, lambda = squared singular values
        out = []
        for b in range(h.size(0)):
            X = h[b][mask[b].bool()]                  # (t, H)
            X = X - X.mean(0, keepdim=True)
            lam = torch.linalg.svdvals(X) ** 2
            out.append((lam.sum() ** 2) / lam.pow(2).sum().clamp(min=1e-12))
        return torch.stack(out)
    raise ValueError(f"unknown geometry metric: {metric}")

## 6. Extract features in batches

In [ ]:
import numpy as np
from tqdm.auto import tqdm

n_hidden = num_layers + 1        # length of the hidden_states tuple (index 0 = embeddings)

def norm_layer(layer):
    """Resolve a possibly-negative layer index to its absolute position in hidden_states."""
    return layer if layer >= 0 else n_hidden + layer

def pool_key(f):
    return f"L{norm_layer(f['layer'])}_{f['pooling']}"

def geom_key(g):
    return f"L{norm_layer(g['layer'])}_{g['metric']}"

features = CONFIG["features"]
geo = CONFIG.get("geometry", [])
pooled = {pool_key(f): [] for f in features}
geoms = {geom_key(g): [] for g in geo}

bs = CONFIG["batch_size"]
for start in tqdm(range(0, len(problems), bs)):
    batch = problems[start:start + bs]
    enc = tokenizer(
        batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=CONFIG["max_input_tokens"],
    ).to(device)

    with torch.no_grad():
        out = model(**enc)

    hs = out.hidden_states                # tuple: (num_layers+1) x (B, T, H)
    mask = enc["attention_mask"]
    for f in features:
        vec = pool(hs[f["layer"]], mask, f["pooling"])
        pooled[pool_key(f)].append(vec.float().cpu().numpy())
    for g in geo:
        s = geometry(hs[g["layer"]], mask, g["metric"])
        geoms[geom_key(g)].append(s.float().cpu().numpy())

pooled_np = {k: np.concatenate(v, axis=0) for k, v in pooled.items()}
geoms_np = {k: np.concatenate(v, axis=0) for k, v in geoms.items()}
for k, v in pooled_np.items():
    print(f"[pooled] {k}: {v.shape}")
for k, v in geoms_np.items():
    print(f"[geom]   {k}: {v.shape}")

## 7. Assemble & save the dataset

Parquet: metadata + one list-valued column per pooled feature + one scalar column per geometry metric.

In [7]:
import pandas as pd

df = pd.DataFrame({
    "problem": [row["problem"] for row in ds],
    "level": [row.get("level") for row in ds],
    "type": [row.get("type") for row in ds],
})
for k, v in pooled_np.items():
    df[k] = list(v)          # each cell = 1-D float array (hidden_size,)
for k, v in geoms_np.items():
    df[k] = v                # scalar per row

df.to_parquet(CONFIG["output_path"], index=False)
print(f"Saved {len(df)} rows -> {CONFIG['output_path']}")
df.head(3)

Saved 200 rows -> hidden_states_dataset.parquet


,problem,level,type,L14_mean,L21_mean,L-1_last,L14_l2_norm,L14_token_norm_std,L21_mean_token_norm,L21_effective_rank,L-1_l2_norm,L-1_anisotropy
0,Square A and Square B are both $2009$ by $2009...,Level 5,Algebra,"[-0.3203125, -0.13085938, 0.73046875, 0.146484...","[-0.796875, 0.48828125, 1.453125, 2.734375, -0...","[-0.7109375, -1.8671875, -0.38476562, -3.5625,...",110.657913,737.444885,241.542633,1.080285,31.496088,0.102854
1,The height (in meters) of a shot cannonball fo...,Level 5,Algebra,"[-0.55078125, 0.13964844, -0.875, -0.95703125,...","[-1.0546875, 1.75, 1.375, 0.828125, 2.203125, ...","[-0.7109375, -0.07714844, -0.41210938, -1.0468...",136.147537,776.073914,274.599365,1.072461,32.835716,0.105976
2,What value of $x$ will give the minimum value ...,Level 3,Algebra,"[-2.734375, -1.109375, -0.091308594, 0.8046875...","[-3.4375, -3.421875, 2.640625, 3.6875, 0.89062...","[0.048828125, -0.38476562, 0.123535156, 2.6562...",308.318756,1340.895508,458.837402,1.026707,40.172054,0.172285


In [8]:
# Optional: download from Colab
# from google.colab import files
# files.download(CONFIG["output_path"])